# Visualize how MFA Gaussians tile a toy manifold

This notebook loads a completed toy-manifold MFA, ARD, or HDDC run and visualizes one planted manifold instance. The shared section only loads and validates artifacts. PCA and UMAP are deliberately independent sections: either section can be deleted without changing the other.

Each method produces two compact, vertically stacked Plotly views. The parameter-only view contains Gaussians whose ambient-space means are within `MAX_MEAN_TO_MANIFOLD_DISTANCE` of the selected manifold, including nearby components that receive no hard assignments. The saved-assignment view contains only components that receive points from the selected manifold. Set `N_COMPONENTS` to 2 for ellipses or 3 for ellipsoids. Solid, dashed, and (in 3D) dotted segments show the projected ambient covariance PCs.

## 1. Choose a trained run

The run must contain a model and its saved responsibility assignments.

In [11]:
from pathlib import Path

RUN_DIR = Path(
"/orfeo/cephfs/scratch/dssc/zenocosini/dalg-cache/toy_manifold_models_30k/adaptive_q_toy_30k_hddc_shared_b_active_set_threshold_sweep/hddc__toy_manifolds_d128_30k_noise1e4__l00__k300__q16__s42__ee0d57aa"
)

## 2. Load and validate the run

This section verifies the run, toy-manifold source, assignment bundle, and canonical row alignment before loading the model parameters.

In [12]:
import json
import sys

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from torch.utils.data import DataLoader

REPO = Path.cwd().resolve()
if not (REPO / "src").is_dir():
    REPO = REPO.parent.resolve()
if not (REPO / "src").is_dir():
    raise RuntimeError("Run this notebook from the repository root or notebooks/.")
sys.path.insert(0, str(REPO / "src"))

from dalg.evaluation.toy_manifold_tiling import _load_model
from dalg.data.shard_activations import ActivationBatchDataset, load_meta_index
from dalg.data.subset_spec import resolve_spec_positions, split_shard_dir_spec

RUN_DIR = RUN_DIR.expanduser()
if not RUN_DIR.is_absolute():
    RUN_DIR = (REPO / RUN_DIR).resolve()

RUN_SPEC_PATH = RUN_DIR / "run_spec.json"
RUN_CONFIG_PATH = RUN_DIR / "config.json"
MODEL_PATH = RUN_DIR / "mfa_model.pt"
ASSIGNMENTS_PATH = RUN_DIR / "mfa_model_assignments.pt"
required = [RUN_SPEC_PATH, RUN_CONFIG_PATH, MODEL_PATH, ASSIGNMENTS_PATH]
missing = [str(path) for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError(f"Missing required run artifacts: {missing}")

run_spec = json.loads(RUN_SPEC_PATH.read_text())
run_config = json.loads(RUN_CONFIG_PATH.read_text())
model_kind = run_spec.get("training", {}).get("model_kind")
if model_kind not in {"mfa", "ard", "hddc"}:
    raise ValueError(f"Unsupported model kind in run_spec.json: {model_kind!r}")
dataset_spec = run_spec.get("dataset", {})
if "shard_dir" not in dataset_spec or "layer" not in dataset_spec:
    raise ValueError("run_spec.json must record dataset.shard_dir and dataset.layer")

shard_dir, subset_spec = split_shard_dir_spec(dataset_spec["shard_dir"])
if not shard_dir.is_absolute():
    shard_dir = (REPO / shard_dir).resolve()
SHARD_CONFIG_PATH = shard_dir / "config.json"
if not SHARD_CONFIG_PATH.is_file():
    raise FileNotFoundError(f"Missing shard config: {SHARD_CONFIG_PATH}")
shard_config = json.loads(SHARD_CONFIG_PATH.read_text())
if shard_config.get("source_kind") != "toy_manifolds":
    raise ValueError("The trained run does not reference toy-manifold shards.")
window = int(shard_config["window"])
drop_prefix = int(shard_config.get("drop_prefix", 0))
if window != 1 or drop_prefix != 0:
    raise ValueError("Toy-manifold visualization expects one activation per row.")

metadata_path = shard_dir / shard_config["manifold_metadata"]
if not metadata_path.is_file():
    raise FileNotFoundError(f"Missing manifold metadata: {metadata_path}")
manifold_metadata = torch.load(metadata_path, map_location="cpu", weights_only=True)
all_manifold_ids = manifold_metadata["row_manifold_ids"].reshape(-1).long()
layer = int(dataset_spec["layer"])
meta_index = load_meta_index(shard_dir, layer=layer)
if all_manifold_ids.numel() != len(meta_index):
    raise ValueError(
        "Toy labels and activation rows are not aligned: "
        f"labels={all_manifold_ids.numel()}, rows={len(meta_index)}"
    )
positions = resolve_spec_positions(
    meta_index, subset_spec, window=window, drop_prefix=drop_prefix
)
position_tensor = torch.as_tensor(positions, dtype=torch.long)
row_manifold_ids = all_manifold_ids[position_tensor]

assignment_bundle = torch.load(
    ASSIGNMENTS_PATH, map_location="cpu", mmap=True, weights_only=True
)
assignments = assignment_bundle["assignments"].reshape(-1).long()
cluster_sizes = assignment_bundle["cluster_sizes"].reshape(-1).long()
if assignment_bundle.get("subset_spec") != subset_spec:
    raise ValueError(
        "Assignment subset does not match the run dataset: "
        f"{assignment_bundle.get('subset_spec')!r} != {subset_spec!r}"
    )
if assignments.numel() != len(positions):
    raise ValueError(
        "Assignments do not cover the selected canonical stream: "
        f"assignments={assignments.numel()}, rows={len(positions)}"
    )

model = _load_model(RUN_DIR, model_kind).cpu().eval()
if int(assignment_bundle["K"]) != model.K or cluster_sizes.numel() != model.K:
    raise ValueError("Assignment K does not match the loaded model.")
if not torch.equal(torch.bincount(assignments, minlength=model.K), cluster_sizes):
    raise ValueError("cluster_sizes is inconsistent with assignments.")
if model.D != int(shard_config["d_model"]):
    raise ValueError("Model dimension does not match the toy-manifold shards.")

dataset = ActivationBatchDataset(
    shard_dir,
    layer=layer,
    row_subset=positions,
    batch_size=4096,
    drop_prefix=drop_prefix,
    dtype=torch.float32,
    shuffle_shards=False,
    shuffle_within_shard=False,
    seed=0,
)
all_points = torch.cat(
    list(DataLoader(dataset, batch_size=None, num_workers=0)), dim=0
).cpu()
if all_points.shape != (len(positions), model.D):
    raise ValueError(
        f"Expected point matrix {(len(positions), model.D)}, got {tuple(all_points.shape)}"
    )
print(
    f"Validated {model_kind.upper()} run {RUN_DIR.name}: "
    f"{len(positions):,} toy points, K={model.K}, D={model.D}, q={model.q}."
)

Validated HDDC run hddc__toy_manifolds_d128_30k_noise1e4__l00__k300__q16__s42__ee0d57aa: 30,000 toy points, K=300, D=128, q=16.


## 3. Inspect and select a manifold

The ID identifies one planted manifold instance. Multiple IDs may share the same type when `manifolds_per_type > 1`.

In [13]:
num_manifolds = int(manifold_metadata["num_manifolds"])
manifold_counts = torch.bincount(row_manifold_ids, minlength=num_manifolds)
manifold_rows = []
for item in manifold_metadata["manifolds"]:
    manifold_id = int(item["manifold_id"])
    manifold_rows.append({
        "manifold_id": manifold_id,
        "type_name": item["type_name"],
        "intrinsic_dim": int(item["intrinsic_dim"]),
        "point_count": int(manifold_counts[manifold_id]),
    })
manifold_table = pd.DataFrame(manifold_rows)
display(manifold_table)
print("ID mapping:", dict(zip(manifold_table.manifold_id, manifold_table.type_name)))

,manifold_id,type_name,intrinsic_dim,point_count
0,0,circle,1,10000
1,1,helix,1,10000
2,2,torus,2,10000


ID mapping: {0: 'circle', 1: 'helix', 2: 'torus'}


In [14]:
# Shared visualization settings. Projection-specific settings live in their sections.
MANIFOLD_ID = 2
N_COMPONENTS = 3          # choose 2 for ellipses or 3 for ellipsoids
ELLIPSE_N_STD = 0.8       # Mahalanobis radius of every Gaussian footprint
MAX_MEAN_TO_MANIFOLD_DISTANCE = 0.1  # ambient Euclidean distance used to prune tiles
MAX_PLOT_POINTS = 10_000  # only plotting is subsampled; projection fitting uses all points
POINT_SIZE = 5
POINT_ALPHA = 0.70
ELLIPSE_ALPHA = 0.08
ANNOTATE_COMPONENT_IDS = False
PLOT_WIDTH = 780
PLOT_PANEL_HEIGHT = 470
RANDOM_SEED = 0

if N_COMPONENTS not in (2, 3):
    raise ValueError("N_COMPONENTS must be 2 or 3.")
if ELLIPSE_N_STD <= 0:
    raise ValueError("ELLIPSE_N_STD must be positive.")
if MAX_MEAN_TO_MANIFOLD_DISTANCE <= 0:
    raise ValueError("MAX_MEAN_TO_MANIFOLD_DISTANCE must be positive.")
if not 0 <= MANIFOLD_ID < num_manifolds:
    raise ValueError(f"MANIFOLD_ID must lie in [0, {num_manifolds - 1}].")

## 4. Shared model geometry and plotting utilities

This is the only preparation used by both projections. It constructs the full Gaussian covariance $\Sigma_k = W_kW_k^\top + \Psi_k$ and extracts its top `N_COMPONENTS` ambient-space eigenvectors.

In [15]:
manifold_mask = row_manifold_ids == MANIFOLD_ID
manifold_points = all_points[manifold_mask].contiguous()
manifold_assignments = assignments[manifold_mask].contiguous()
if manifold_points.numel() == 0:
    raise ValueError(f"Selected manifold {MANIFOLD_ID} has no points.")

with torch.no_grad():
    means = model.mu.detach().cpu().float()
    loadings = model._W().detach().cpu().float()
    psi = model._psi().detach().cpu().float()
    covariances = torch.bmm(loadings, loadings.transpose(1, 2)) + torch.diag_embed(psi)
    covariance_eigenvalues, covariance_eigenvectors = torch.linalg.eigh(covariances)
    ambient_pc_values = torch.flip(
        covariance_eigenvalues[:, -N_COMPONENTS:], dims=(1,)
    )
    ambient_pc_vectors = torch.flip(
        covariance_eigenvectors[:, :, -N_COMPONENTS:], dims=(2,)
    )
    scales = model._scale().detach().cpu()
    rank_mask = getattr(model, "rank_mask", None)
    if rank_mask is not None:
        scales = scales * rank_mask.detach().cpu()
    noise_floor = psi.mean(dim=1, keepdim=True)
    effective_ranks = (scales.square() > noise_floor).sum(dim=1).long()
    mixture_weights = model.pi_logits.softmax(dim=0).detach().cpu()

means_np = means.numpy()
loadings_np = loadings.numpy()
psi_np = psi.numpy()
covariances_np = covariances.numpy()
ambient_pc_values_np = ambient_pc_values.numpy()
ambient_pc_vectors_np = ambient_pc_vectors.numpy()
manifold_points_np = manifold_points.numpy()
manifold_assignments_np = manifold_assignments.numpy()
active_components = np.unique(manifold_assignments_np)

points_here = torch.bincount(manifold_assignments, minlength=model.K)
mean_to_manifold_distances = torch.cdist(means, manifold_points).amin(dim=1)
plot_components = torch.nonzero(
    mean_to_manifold_distances <= MAX_MEAN_TO_MANIFOLD_DISTANCE
).flatten().numpy()
if len(plot_components) == 0:
    raise ValueError(
        "No Gaussian means are within MAX_MEAN_TO_MANIFOLD_DISTANCE of the selected manifold."
    )
purity = points_here.float() / cluster_sizes.float().clamp_min(1)
component_table = pd.DataFrame({
    "component": active_components,
    "points_on_manifold": points_here[active_components].numpy(),
    "total_assigned_points": cluster_sizes[active_components].numpy(),
    "mean_to_manifold_distance": mean_to_manifold_distances[active_components].numpy(),
    "purity": purity[active_components].numpy(),
    "effective_rank": effective_ranks[active_components].numpy(),
    "mixture_weight": mixture_weights[active_components].numpy(),
}).sort_values("component").reset_index(drop=True)
display(component_table)
print(
    f"Selected manifold {MANIFOLD_ID} ({manifold_rows[MANIFOLD_ID]['type_name']}): "
    f"{len(manifold_points_np):,} points assigned to {len(active_components)} components; "
    f"plotting {len(plot_components)} of {model.K} components whose mean is within "
    f"{MAX_MEAN_TO_MANIFOLD_DISTANCE:g} of the manifold."
)

,component,points_on_manifold,total_assigned_points,mean_to_manifold_distance,purity,effective_rank,mixture_weight
0,1,99,99,0.070880,1.0,3,0.003316
1,2,48,48,0.052024,1.0,3,0.003316
2,5,25,25,0.077684,1.0,3,0.003316
3,10,95,95,0.048110,1.0,3,0.003316
4,11,77,77,0.060939,1.0,3,0.003316
...,...,...,...,...,...,...,...
180,291,20,20,0.088161,1.0,3,0.003316
181,292,85,85,0.059937,1.0,3,0.003316
182,293,9,9,0.070489,1.0,3,0.003316
183,294,42,42,0.045826,1.0,3,0.003316


Selected manifold 2 (torus): 10,000 points assigned to 185 components; plotting 191 of 300 components whose mean is within 0.1 of the manifold.


In [16]:
import plotly.graph_objects as go
from plotly.colors import sample_colorscale
from plotly.subplots import make_subplots

COMPONENT_COLORS = sample_colorscale(
    "Turbo", np.linspace(0.02, 0.98, model.K)
)


def component_color(component_id, alpha=None):
    color = COMPONENT_COLORS[int(component_id)]
    if alpha is None:
        return color
    return color.replace("rgb(", "rgba(").replace(")", f", {alpha})")


def covariance_eigendecomposition(covariance):
    covariance = np.asarray(covariance, dtype=np.float64)
    covariance = 0.5 * (covariance + covariance.T)
    values, vectors = np.linalg.eigh(covariance)
    if values.min() <= 0:
        raise ValueError("Projected covariance is not positive definite.")
    return covariance, values, vectors


def covariance_precision(covariance):
    covariance, values, vectors = covariance_eigendecomposition(covariance)
    precision = (vectors / values[None, :]) @ vectors.T
    return covariance, precision


def ellipse_polygon(mean, covariance, n_std, num_points=100):
    mean = np.asarray(mean, dtype=np.float64)
    _, values, vectors = covariance_eigendecomposition(covariance)
    transform = vectors @ np.diag(n_std * np.sqrt(values))
    angles = np.linspace(0.0, 2.0 * np.pi, num_points)
    circle = np.stack([np.cos(angles), np.sin(angles)])
    return mean[:, None] + transform @ circle


def ellipsoid_mesh(mean, covariance, n_std, num_u=18, num_v=10):
    mean = np.asarray(mean, dtype=np.float64)
    _, values, vectors = covariance_eigendecomposition(covariance)
    transform = vectors @ np.diag(n_std * np.sqrt(values))
    u = np.linspace(0.0, 2.0 * np.pi, num_u, endpoint=False)
    v = np.linspace(0.0, np.pi, num_v)
    uu, vv = np.meshgrid(u, v)
    sphere = np.stack([
        np.cos(uu) * np.sin(vv),
        np.sin(uu) * np.sin(vv),
        np.cos(vv),
    ]).reshape(3, -1)
    vertices = mean[:, None] + transform @ sphere
    face_i, face_j, face_k = [], [], []
    for row_index in range(num_v - 1):
        for column_index in range(num_u):
            next_column = (column_index + 1) % num_u
            top_left = row_index * num_u + column_index
            top_right = row_index * num_u + next_column
            bottom_left = (row_index + 1) * num_u + column_index
            bottom_right = (row_index + 1) * num_u + next_column
            face_i.extend([top_left, top_right])
            face_j.extend([top_right, bottom_right])
            face_k.extend([bottom_left, bottom_left])
    return vertices, face_i, face_j, face_k


def clipped_axis_segment(mean, covariance, direction, n_std):
    mean = np.asarray(mean, dtype=np.float64)
    direction = np.asarray(direction, dtype=np.float64)
    direction_norm = np.linalg.norm(direction)
    if direction_norm < 1e-12:
        return np.stack([mean, mean])
    unit = direction / direction_norm
    _, precision = covariance_precision(covariance)
    half_length = n_std / np.sqrt(max(float(unit @ precision @ unit), 1e-12))
    return np.stack([mean - half_length * unit, mean + half_length * unit])


def validate_projected_tiles(projected_means, projected_covariances, projected_pc_directions):
    expected_means = (model.K, N_COMPONENTS)
    expected_covariances = (model.K, N_COMPONENTS, N_COMPONENTS)
    expected_directions = (model.K, N_COMPONENTS, N_COMPONENTS)
    if projected_means.shape != expected_means:
        raise ValueError(f"Expected projected means {expected_means}, got {projected_means.shape}.")
    if projected_covariances.shape != expected_covariances:
        raise ValueError(f"Expected projected covariances {expected_covariances}, got {projected_covariances.shape}.")
    if projected_pc_directions.shape != expected_directions:
        raise ValueError(f"Expected projected PC directions {expected_directions}, got {projected_pc_directions.shape}.")
    if not (
        np.isfinite(projected_means).all()
        and np.isfinite(projected_covariances).all()
        and np.isfinite(projected_pc_directions).all()
    ):
        raise ValueError("Projected tile geometry contains non-finite values.")
    for component_id in range(model.K):
        covariance, precision = covariance_precision(projected_covariances[component_id])
        for pc_number in range(N_COMPONENTS):
            segment = clipped_axis_segment(
                projected_means[component_id], covariance,
                projected_pc_directions[component_id, pc_number], ELLIPSE_N_STD,
            )
            for endpoint in segment:
                delta = endpoint - np.asarray(projected_means[component_id], dtype=np.float64)
                radius_squared = float(delta @ precision @ delta)
                if radius_squared > ELLIPSE_N_STD ** 2 * (1.0 + 1e-4):
                    raise ValueError("A projected PC segment exceeds its Gaussian footprint.")


def add_footprints(fig, projected_means, projected_covariances, component_ids, row):
    for component_id in component_ids:
        component_id = int(component_id)
        mean = projected_means[component_id]
        covariance = projected_covariances[component_id]
        points_on_manifold = int(points_here[component_id])
        total_assigned_points = int(cluster_sizes[component_id])
        mean_distance = float(mean_to_manifold_distances[component_id])
        if N_COMPONENTS == 2:
            polygon = ellipse_polygon(mean, covariance, ELLIPSE_N_STD)
            trace = go.Scatter(
                x=polygon[0], y=polygon[1], mode="lines",
                line=dict(color=component_color(component_id, 0.62), width=1),
                fill="toself", fillcolor=component_color(component_id, ELLIPSE_ALPHA),
                customdata=np.tile(
                    [component_id, points_on_manifold, total_assigned_points, mean_distance],
                    (len(polygon[0]), 1),
                ),
                hovertemplate=(
                    "component %{customdata[0]:.0f}<br>"
                    "points on selected manifold %{customdata[1]:,.0f}<br>"
                    "total assigned points %{customdata[2]:,.0f}<br>"
                    "mean distance to manifold %{customdata[3]:.5f}<extra></extra>"
                ),
                showlegend=False,
            )
        else:
            vertices, face_i, face_j, face_k = ellipsoid_mesh(mean, covariance, ELLIPSE_N_STD)
            trace = go.Mesh3d(
                x=vertices[0], y=vertices[1], z=vertices[2],
                i=face_i, j=face_j, k=face_k,
                color=component_color(component_id), opacity=ELLIPSE_ALPHA,
                flatshading=False,
                hovertemplate=(
                    f"component {component_id}<br>"
                    f"points on selected manifold {points_on_manifold:,}<br>"
                    f"total assigned points {total_assigned_points:,}<br>"
                    f"mean distance to manifold {mean_distance:.5f}<extra></extra>"
                ),
                showscale=False, showlegend=False,
            )
        fig.add_trace(trace, row=row, col=1)


def add_pc_axes(fig, projected_means, projected_covariances, projected_pc_directions, component_ids, row):
    styles = (
        ("ambient PC1", "solid", "rgba(20, 20, 20, 0.95)", 2.4),
        ("ambient PC2", "dash", "rgba(210, 55, 55, 0.95)", 2.0),
        ("ambient PC3", "dot", "rgba(35, 95, 210, 0.95)", 1.8),
    )
    trace_class = go.Scattergl if N_COMPONENTS == 2 else go.Scatter3d
    for pc_number, (name, dash, color, width) in enumerate(styles[:N_COMPONENTS]):
        coordinates = [[] for _ in range(N_COMPONENTS)]
        for component_id in component_ids:
            component_id = int(component_id)
            segment = clipped_axis_segment(
                projected_means[component_id], projected_covariances[component_id],
                projected_pc_directions[component_id, pc_number], ELLIPSE_N_STD,
            )
            for dimension in range(N_COMPONENTS):
                coordinates[dimension].extend([segment[0, dimension], segment[1, dimension], None])
        trace_arguments = dict(
            x=coordinates[0], y=coordinates[1], mode="lines",
            line=dict(color=color, width=width, dash=dash), name=name,
            legendgroup=f"pc{pc_number + 1}", showlegend=(row == 1), hoverinfo="skip",
        )
        if N_COMPONENTS == 3:
            trace_arguments["z"] = coordinates[2]
        fig.add_trace(trace_class(**trace_arguments), row=row, col=1)


def add_means(fig, projected_means, component_ids, row):
    component_ids = np.asarray(component_ids, dtype=int)
    selected_means = projected_means[component_ids]
    customdata = np.column_stack([
        component_ids, effective_ranks[component_ids].numpy(),
        mixture_weights[component_ids].numpy(), points_here[component_ids].numpy(),
        cluster_sizes[component_ids].numpy(), mean_to_manifold_distances[component_ids].numpy(),
    ])
    trace_class = go.Scattergl if N_COMPONENTS == 2 else go.Scatter3d
    trace_arguments = dict(
        x=selected_means[:, 0], y=selected_means[:, 1],
        mode="markers+text" if ANNOTATE_COMPONENT_IDS else "markers",
        text=[str(component_id) for component_id in component_ids] if ANNOTATE_COMPONENT_IDS else None,
        textposition="top center", textfont=dict(size=9),
        marker=dict(
            size=7, symbol="x",
            color=[component_color(component_id) for component_id in component_ids],
            line=dict(width=1),
        ),
        customdata=customdata,
        hovertemplate=(
            "component %{customdata[0]:.0f}<br>effective rank %{customdata[1]:.0f}<br>"
            "mixture weight %{customdata[2]:.5f}<br>"
            "points on selected manifold %{customdata[3]:,.0f}<br>"
            "total assigned points %{customdata[4]:,.0f}<br>"
            "mean distance to manifold %{customdata[5]:.5f}<br>"
            "mean (%{x:.4f}, %{y:.4f})<extra></extra>"
        ),
        showlegend=False,
    )
    if N_COMPONENTS == 3:
        trace_arguments["z"] = selected_means[:, 2]
        trace_arguments["hovertemplate"] = (
            "component %{customdata[0]:.0f}<br>effective rank %{customdata[1]:.0f}<br>"
            "mixture weight %{customdata[2]:.5f}<br>"
            "points on selected manifold %{customdata[3]:,.0f}<br>"
            "total assigned points %{customdata[4]:,.0f}<br>"
            "mean distance to manifold %{customdata[5]:.5f}<br>"
            "mean (%{x:.4f}, %{y:.4f}, %{z:.4f})<extra></extra>"
        )
    fig.add_trace(trace_class(**trace_arguments), row=row, col=1)


def add_points(fig, projected_points, shown_assignments, row):
    trace_class = go.Scattergl if N_COMPONENTS == 2 else go.Scatter3d
    trace_arguments = dict(
        x=projected_points[:, 0], y=projected_points[:, 1],
        mode="markers", showlegend=False,
    )
    if N_COMPONENTS == 3:
        trace_arguments["z"] = projected_points[:, 2]
    if shown_assignments is None:
        trace_arguments["marker"] = dict(size=POINT_SIZE, color="rgba(55, 55, 55, 0.70)")
        trace_arguments["hovertemplate"] = (
            "point (%{x:.4f}, %{y:.4f})<extra></extra>" if N_COMPONENTS == 2
            else "point (%{x:.4f}, %{y:.4f}, %{z:.4f})<extra></extra>"
        )
    else:
        trace_arguments["marker"] = dict(
            size=POINT_SIZE, color=shown_assignments, colorscale="Turbo",
            cmin=0, cmax=max(model.K - 1, 1), opacity=POINT_ALPHA,
            colorbar=dict(title="component ID", len=0.38, y=0.20, x=1.02),
        )
        trace_arguments["customdata"] = np.column_stack([
            shown_assignments,
            points_here.numpy()[shown_assignments],
            cluster_sizes.numpy()[shown_assignments],
        ])
        trace_arguments["hovertemplate"] = (
            "component %{customdata[0]:.0f}<br>"
            "points on selected manifold %{customdata[1]:,.0f}<br>"
            "total assigned points %{customdata[2]:,.0f}<br>"
            "point (%{x:.4f}, %{y:.4f})<extra></extra>"
            if N_COMPONENTS == 2 else (
                "component %{customdata[0]:.0f}<br>"
                "points on selected manifold %{customdata[1]:,.0f}<br>"
                "total assigned points %{customdata[2]:,.0f}<br>"
                "point (%{x:.4f}, %{y:.4f}, %{z:.4f})<extra></extra>"
            )
        )
    fig.add_trace(trace_class(**trace_arguments), row=row, col=1)


def plot_projection_pair(
    projected_points, projected_means, projected_covariances, projected_pc_directions, method_name, note
):
    rng = np.random.default_rng(RANDOM_SEED)
    if len(projected_points) > MAX_PLOT_POINTS:
        shown = np.sort(rng.choice(len(projected_points), MAX_PLOT_POINTS, replace=False))
    else:
        shown = np.arange(len(projected_points))
    shown_points = projected_points[shown]
    shown_assignments = manifold_assignments_np[shown]

    low = projected_points.min(axis=0)
    high = projected_points.max(axis=0)
    margin = np.maximum((high - low) * 0.08, 1e-6)
    limits = (low - margin, high + margin)
    subplot_type = "xy" if N_COMPONENTS == 2 else "scene"
    fig = make_subplots(
        rows=2, cols=1,
        specs=[[{"type": subplot_type}], [{"type": subplot_type}]],
        subplot_titles=(
            f"Parameters only: {len(plot_components)} nearby Gaussian tiles",
            f"Saved assignments: {len(active_components)} active components",
        ),
        vertical_spacing=0.10,
    )

    add_footprints(fig, projected_means, projected_covariances, plot_components, row=1)
    add_footprints(fig, projected_means, projected_covariances, active_components, row=2)
    point_trace_indices = []
    for point_assignments, row in ((None, 1), (shown_assignments, 2)):
        point_trace_indices.append(len(fig.data))
        add_points(fig, shown_points, shown_assignments=point_assignments, row=row)
    add_pc_axes(
        fig, projected_means, projected_covariances, projected_pc_directions,
        plot_components, row=1,
    )
    add_pc_axes(
        fig, projected_means, projected_covariances, projected_pc_directions,
        active_components, row=2,
    )
    add_means(fig, projected_means, plot_components, row=1)
    add_means(fig, projected_means, active_components, row=2)

    if N_COMPONENTS == 2:
        for row, x_anchor in ((1, "x"), (2, "x2")):
            fig.update_xaxes(
                range=[limits[0][0], limits[1][0]], title_text=f"{method_name} 1",
                showgrid=True, gridcolor="rgba(0, 0, 0, 0.08)", zeroline=False,
                row=row, col=1,
            )
            fig.update_yaxes(
                range=[limits[0][1], limits[1][1]], title_text=f"{method_name} 2",
                showgrid=True, gridcolor="rgba(0, 0, 0, 0.08)", zeroline=False,
                scaleanchor=x_anchor, scaleratio=1, row=row, col=1,
            )
    else:
        for scene_name in ("scene", "scene2"):
            fig.update_layout(**{
                scene_name: dict(
                    xaxis=dict(range=[limits[0][0], limits[1][0]], title=f"{method_name} 1"),
                    yaxis=dict(range=[limits[0][1], limits[1][1]], title=f"{method_name} 2"),
                    zaxis=dict(range=[limits[0][2], limits[1][2]], title=f"{method_name} 3"),
                    aspectmode="data", bgcolor="white",
                )
            })

    fig.update_layout(
        template="plotly_white", width=PLOT_WIDTH,
        height=2 * PLOT_PANEL_HEIGHT + 180,
        title=dict(
            text=(
                f"{method_name}: manifold {MANIFOLD_ID} "
                f"({manifold_rows[MANIFOLD_ID]['type_name']})<br><sup>{note}</sup>"
            ),
            x=0.5,
        ),
        legend=dict(orientation="h", x=0.5, xanchor="center", y=1.01, yanchor="bottom"),
        updatemenus=[dict(
            type="buttons", direction="right", active=0,
            x=0.0, xanchor="left", y=1.08, yanchor="bottom",
            buttons=[
                dict(
                    label="Show dataset points", method="restyle",
                    args=[{"visible": True}, point_trace_indices],
                ),
                dict(
                    label="Gaussians only", method="restyle",
                    args=[{"visible": False}, point_trace_indices],
                ),
            ],
        )],
        margin=dict(l=55, r=85, t=125, b=45), hovermode="closest",
    )
    fig.show(config=dict(scrollZoom=True, responsive=True, displaylogo=False))
    return fig


# PCA projection

This section is self-contained. PCA is fitted only on the selected manifold. Because PCA is linear, every plotted covariance is the exact projected covariance $P\Sigma_kP^\top$. Delete this section if PCA is not needed.

In [17]:
RUN_PCA = True
PCA_SVD_SOLVER = "full"

In [18]:
if RUN_PCA:
    from sklearn.decomposition import PCA

    pca = PCA(n_components=N_COMPONENTS, svd_solver=PCA_SVD_SOLVER)
    pca_points = pca.fit_transform(manifold_points_np)
    pca_means = pca.transform(means_np)
    pca_basis = pca.components_
    pca_covariances = np.einsum(
        "pd,kde,qe->kpq", pca_basis, covariances_np, pca_basis, optimize=True
    )
    pca_pc_directions = np.einsum(
        "pd,kdr->krp", pca_basis, ambient_pc_vectors_np, optimize=True
    )
    validate_projected_tiles(pca_means, pca_covariances, pca_pc_directions)

    display(pd.DataFrame({
        "PCA axis": np.arange(1, N_COMPONENTS + 1),
        "explained_variance_ratio": pca.explained_variance_ratio_,
        "cumulative": np.cumsum(pca.explained_variance_ratio_),
    }))
    pca_figure = plot_projection_pair(
        pca_points,
        pca_means,
        pca_covariances,
        pca_pc_directions,
        method_name="PCA",
        note=(
            f"exact projected {ELLIPSE_N_STD:g}σ covariances; "
            "solid=ambient PC1, dashed=ambient PC2"
            + (", dotted=ambient PC3" if N_COMPONENTS == 3 else "")
        ),
    )

,PCA axis,explained_variance_ratio,cumulative
0,1,0.454720,0.454720
1,2,0.444976,0.899697
2,3,0.100303,1.000000


# UMAP projection

This section is self-contained and imports `umap` only when enabled. UMAP is nonlinear, so a Gaussian has no exact projected covariance. The footprints below are deterministic, sample-based second-moment summaries around the transformed model means. Delete this section and the `umap-learn` dependency if UMAP is not needed.

In [19]:
RUN_UMAP = False
UMAP_N_NEIGHBORS = 30
UMAP_MIN_DIST = 0.10
UMAP_METRIC = "euclidean"
UMAP_GAUSSIAN_SAMPLES = 256
UMAP_COMPONENT_BATCH = 20
UMAP_COVARIANCE_JITTER = 1e-5

In [20]:
if RUN_UMAP:
    import umap

    if UMAP_GAUSSIAN_SAMPLES < 3:
        raise ValueError("UMAP_GAUSSIAN_SAMPLES must be at least 3.")
    if UMAP_COMPONENT_BATCH <= 0:
        raise ValueError("UMAP_COMPONENT_BATCH must be positive.")

    umap_reducer = umap.UMAP(
        n_components=N_COMPONENTS,
        n_neighbors=UMAP_N_NEIGHBORS,
        min_dist=UMAP_MIN_DIST,
        metric=UMAP_METRIC,
        random_state=RANDOM_SEED,
        transform_seed=RANDOM_SEED,
        n_jobs=1,
    )
    umap_points = umap_reducer.fit_transform(manifold_points_np)
    umap_means = umap_reducer.transform(means_np)

    # Estimate each nonlinear projected footprint using only Gaussian parameters.
    umap_covariances = np.empty((model.K, N_COMPONENTS, N_COMPONENTS), dtype=np.float64)
    umap_rng = np.random.default_rng(RANDOM_SEED)
    for start in range(0, model.K, UMAP_COMPONENT_BATCH):
        end = min(start + UMAP_COMPONENT_BATCH, model.K)
        count = end - start
        latent = umap_rng.standard_normal(
            (count, UMAP_GAUSSIAN_SAMPLES, model.q), dtype=np.float32
        )
        noise = umap_rng.standard_normal(
            (count, UMAP_GAUSSIAN_SAMPLES, model.D), dtype=np.float32
        )
        samples = (
            means_np[start:end, None, :]
            + np.einsum("kdq,ksq->ksd", loadings_np[start:end], latent, optimize=True)
            + np.sqrt(psi_np[start:end])[:, None, :] * noise
        )
        embedded = umap_reducer.transform(samples.reshape(-1, model.D)).reshape(
            count, UMAP_GAUSSIAN_SAMPLES, N_COMPONENTS
        )
        delta = embedded - umap_means[start:end, None, :]
        umap_covariances[start:end] = (
            np.einsum("ksi,ksj->kij", delta, delta, optimize=True)
            / (UMAP_GAUSSIAN_SAMPLES - 1)
        )
    umap_covariances += UMAP_COVARIANCE_JITTER * np.eye(N_COMPONENTS)[None]

    # Project μ ± sqrt(λ_i) v_i and use the resulting secant as ambient PC i's UMAP direction.
    pc_queries = []
    for component_id in range(model.K):
        for pc_number in range(N_COMPONENTS):
            step = (
                np.sqrt(max(float(ambient_pc_values_np[component_id, pc_number]), 0.0))
                * ambient_pc_vectors_np[component_id, :, pc_number]
            )
            pc_queries.extend([means_np[component_id] - step, means_np[component_id] + step])
    embedded_pc_queries = umap_reducer.transform(np.stack(pc_queries)).reshape(
        model.K, N_COMPONENTS, 2, N_COMPONENTS
    )
    umap_pc_directions = embedded_pc_queries[:, :, 1] - embedded_pc_queries[:, :, 0]
    validate_projected_tiles(umap_means, umap_covariances, umap_pc_directions)

    umap_figure = plot_projection_pair(
        umap_points,
        umap_means,
        umap_covariances,
        umap_pc_directions,
        method_name="UMAP",
        note=(
            f"sample-based approximate {ELLIPSE_N_STD:g}σ footprints; "
            "solid=ambient PC1, dashed=ambient PC2"
            + (", dotted=ambient PC3" if N_COMPONENTS == 3 else "")
        ),
    )

## Reading the plots

A useful tiling should divide the selected manifold into coherent assignment-colored regions. In the parameter-only panel, Gaussian tiles are selected by the ambient Euclidean distance from their mean to the nearest point on the selected manifold. In the saved-assignment panel, tiles are selected by hard assignments and therefore exclude inactive components. The crosses are transformed model means. Solid PC1 segments should generally follow the local tangent; dashed PC2 segments expose the next-largest covariance direction. In 3D, dotted blue segments show PC3. Every segment is clipped to the displayed ellipse or ellipsoid rather than using an arbitrary visual length.

PCA ellipses/ellipsoids are exact marginal covariances in the selected PCA subspace. UMAP footprints summarize transformed parameter samples and must not be interpreted as exact Gaussian confidence regions. Use the **Show dataset points** / **Gaussians only** buttons above the plot to toggle both panels' point clouds. Hover over means, footprints, or assignment-colored points for component details, including the number of points from the selected manifold and the total number assigned to that component; drag to pan or orbit and use the wheel to zoom.